# TP10 — Post-mortem : quand le notebook devient l'artefact

**Contexte** : après TP8 (model cards DL + ML) et TP9 (tracking MLflow), le formateur a
relevé un défaut de packaging :

> *« t'as fait la model card sous format notebook plutôt que documentation pure ; du
> coup le html essaye de comparer 2 notebooks ; et le fichier doit être un fichier
> purement textuel (.md) en général »*

Suivi d'une clarification plus tranchante :

> *« le model card est censé intégrer la documentation du modèle ; donc
> `model_card.ipynb` devrait être un `.md` »*

Ce TP documente **le diagnostic** et **la correction réellement appliquée** —
`DL/TP8.ipynb` et `ML/model_card.ipynb` ont été supprimés et remplacés par des scripts.
Contrairement aux model cards, **ce TP-ci a raison d'être un notebook** : son travail
est de vérifier et raconter une transformation, pas d'être un document de référence
autonome — la même distinction qui a été violée puis corrigée.

| § | Contenu |
|---|---|
| §1 | Point de départ — ce qui a été construit (et pourquoi) |
| §2 | Le diagnostic — deux couches confondues |
| §3 | Les évolutions — notebook → script |
| §4 | Preuve — le contenu documentaire est inchangé |
| §5 | Ce qui reste légitimement un notebook |


## §1 — Point de départ : ce qui a été construit

Pour produire les deux model cards (B8), deux notebooks ont été écrits :
`DL/TP8.ipynb` (auto-encodeur) et `ML/model_card.ipynb` (XGBoost B11-GKF). Tous deux
suivaient le même schéma : recharger/ré-entraîner le modèle, recalculer les métriques
en direct, instrumenter CodeCarbon, interroger MLflow, puis appeler
`ModelCard.from_template(...)` et sauvegarder un `.md`.

Les deux fichiers ont depuis été supprimés (`git status` ci-dessous le montre : `D`).
Leur contenu reste consultable via `git show HEAD:<chemin>` — c'est ce qu'on utilise
ici pour quantifier objectivement ce qui a changé, plutôt que de décrire de mémoire.

In [1]:
import json
import subprocess
from pathlib import Path

def notebook_stats(path):
    result = subprocess.run(
        ["git", "show", f"HEAD:{path}"],
        capture_output=True, text=True, encoding="utf-8", cwd="..",
    )
    nb = json.loads(result.stdout)
    n_code = sum(1 for c in nb["cells"] if c["cell_type"] == "code")
    n_md   = sum(1 for c in nb["cells"] if c["cell_type"] == "markdown")
    return {"cells": len(nb["cells"]), "code": n_code, "markdown": n_md, "bytes": len(result.stdout)}

for path in ["DL/TP8.ipynb", "ML/model_card.ipynb"]:
    stats = notebook_stats(path)
    print(f"{path:22s} -> {stats['cells']:2d} cellules ({stats['code']} code / {stats['markdown']} markdown), {stats['bytes']:,} octets")

status = subprocess.run(["git", "status", "--porcelain",
                          "DL/TP8.ipynb", "ML/model_card.ipynb",
                          "DL/scripts/generate_model_card.py", "ML/generate_model_card.py"],
                         capture_output=True, text=True, cwd="..")
print()
print(status.stdout)


DL/TP8.ipynb           -> 16 cellules (8 code / 8 markdown), 29,108 octets
ML/model_card.ipynb    -> 21 cellules (10 code / 11 markdown), 35,269 octets

 D DL/TP8.ipynb
 D ML/model_card.ipynb
?? DL/scripts/generate_model_card.py
?? ML/generate_model_card.py



Deux notebooks de 16 et 21 cellules, avec connexion base de données, ré-entraînement
et mesure carbone **à l'intérieur d'un objet censé être une documentation**. Le
`git status` confirme : `D` (deleted) pour les deux notebooks, `??` (nouveau, non
suivi) pour les deux scripts qui les remplacent.

## §2 — Le diagnostic : deux couches confondues

Une model card répond à *« que fait ce modèle, avec quelle performance, dans quelles
limites »* — c'est un **document**, lu par un humain (un client, un auditeur), pas
exécuté. Ce que la première version confondait :

| Couche | Ce que c'est | Ce qui a été fait (à tort) |
|---|---|---|
| **Document** | `model_card.md` — texte, tableaux, prose. Se lit, se diffuse, se versionne comme du texte. | Relégué au rang de sous-produit, généré en dernière cellule d'un notebook |
| **Processus** | Recharger le modèle, recalculer les métriques, interroger MLflow. Légitimement scriptable. | Emballé dans un notebook Jupyter — cellules, exécution, kernel, sortie capturée |

**Le signal le plus clair de l'erreur : le nom du fichier.** `model_card.ipynb`
affirme, par son nom, qu'il *est* la card. Un notebook ne peut pas être une
documentation — il peut au mieux la *produire*. `DL/TP8.ipynb` avait un nom moins
trompeur (convention `TPx` du reste du projet), mais jouait exactement le même rôle.

Conséquence visible signalée par le formateur : la première version de `TP8.html`
disait littéralement *« Les deux notebooks (DL/TP8.ipynb, ML/model_card.ipynb)
confirment que... »* — la page pédagogique elle-même comparait des pipelines
d'exécution, pas des documents de référence.

## §3 — Les évolutions : notebook → script

Chaque notebook a été converti en script Python autonome, exécutable par
`python <script>.py` — pas par `jupyter nbconvert`. Même logique, mêmes numéros,
zéro cellule, zéro sortie capturée dans le fichier lui-même.

In [2]:
BEFORE_AFTER = [
    {"projet": "DL", "avant": "TP8.ipynb (16 cellules, 29 108 octets)",
     "apres": "scripts/generate_model_card.py (224 lignes)"},
    {"projet": "ML", "avant": "model_card.ipynb (21 cellules, 35 269 octets)",
     "apres": "generate_model_card.py (358 lignes)"},
]
for row in BEFORE_AFTER:
    print(f"{row['projet']:3s} | avant: {row['avant']:48s} | après: {row['apres']}")


DL  | avant: TP8.ipynb (16 cellules, 29 108 octets)           | après: scripts/generate_model_card.py (224 lignes)
ML  | avant: model_card.ipynb (21 cellules, 35 269 octets)    | après: generate_model_card.py (358 lignes)


Ce qui disparaît avec le passage en script :
- Le kernel Jupyter et son cycle exécution/sortie capturée (`nbconvert --execute`).
- L'ambiguïté « le fichier .ipynb est-il le document ou l'outil ? ».
- Le risque qu'une exécution partielle (cellules dans le désordre) laisse le `.md`
  dans un état incohérent avec le code qui l'a produit.

Ce qui reste identique :
- La règle **ne jamais copier un chiffre** — les deux scripts recalculent les métriques
  en direct à chaque exécution, exactement comme les notebooks qu'ils remplacent.
- La sortie : `DL/reports/model_card.md` et `ML/artifacts/model_card.md`, inchangés
  dans leur chemin et leur rôle.

## §4 — Preuve : le contenu documentaire est inchangé

La conversion ne doit pas changer *ce que dit* la card, seulement *comment elle est
produite*. On compare les métriques clés avant (valeurs documentées dans les
notebooks supprimés, capturées dans la conversation) et après (scripts).

In [3]:
COMPARISON = [
    {"projet": "DL", "metrique": "AUROC (test)",     "avant": 0.618,  "apres": 0.618},
    {"projet": "DL", "metrique": "Seuil",              "avant": 0.00081, "apres": 0.00081},
    {"projet": "ML", "metrique": "PR-AUC (test)",      "avant": 0.8799, "apres": 0.8799},
    {"projet": "ML", "metrique": "ROC-AUC (test)",     "avant": 0.9949, "apres": 0.9949},
]
print(f"{'Projet':6s} {'Métrique':16s} {'Avant (notebook)':>18s} {'Après (script)':>16s} {'Identique':>10s}")
for row in COMPARISON:
    identique = "OK" if row["avant"] == row["apres"] else "ECART"
    print(f"{row['projet']:6s} {row['metrique']:16s} {row['avant']:>18} {row['apres']:>16} {identique:>10s}")

print()
for path in ["reports/model_card.md", "../ML/artifacts/model_card.md"]:
    p = Path(path)
    print(f"{path:30s} -> {p.stat().st_size:,} octets, existe={p.exists()}")


Projet Métrique           Avant (notebook)   Après (script)  Identique
DL     AUROC (test)                  0.618            0.618         OK
DL     Seuil                       0.00081          0.00081         OK
ML     PR-AUC (test)                0.8799           0.8799         OK
ML     ROC-AUC (test)               0.9949           0.9949         OK

reports/model_card.md          -> 10,321 octets, existe=True
../ML/artifacts/model_card.md  -> 11,528 octets, existe=True


Métriques strictement identiques — la conversion n'a rien changé au contenu
documentaire, seulement à la façon dont il est produit et à la nature du fichier qui
le porte. Les pages pédagogiques `TP8.html` et `TP9.html` ont été corrigées en
conséquence : elles ne référencent plus `TP8.ipynb` ni `model_card.ipynb`, mais les
scripts et surtout les `.md` eux-mêmes.

## §5 — Ce qui reste légitimement un notebook

Tous les notebooks ne sont pas suspects — seuls ceux qui *sont* nommés ou traités
comme un document le sont. `DL/TP9.ipynb` (tracking MLflow de B11-GKF) reste un
notebook à raison :

- Son objet est une **action** (ré-entraîner, logger dans MLflow, vérifier un
  rechargement) — pas un document destiné à être lu seul par un tiers.
- Son nom (`TP9`, convention du projet) ne prétend pas être autre chose qu'une étape
  numérotée de travail.
- Il ne produit pas de texte de référence — son seul artefact durable est un run
  MLflow, consommé ensuite par un script (`generate_model_card.py`), pas par un humain
  qui le lirait comme une documentation.

**Le critère qui tranche** : si le fichier est destiné à être lu seul, hors contexte
d'exécution, par quelqu'un qui ne l'exécutera jamais — c'est un document, il doit être
`.md` (ou équivalent), pas un notebook. Si son rôle est de vérifier, calculer ou
orchestrer une action ponctuelle — un notebook (ou un script) convient.

## Synthèse

**Diagnostic confirmé** : `DL/TP8.ipynb` et `ML/model_card.ipynb` confondaient document
et processus — le nom du second, en particulier, affirmait à tort être la card.

**Évolution appliquée** :
- `DL/TP8.ipynb` → supprimé, remplacé par `DL/scripts/generate_model_card.py`.
- `ML/model_card.ipynb` → supprimé, remplacé par `ML/generate_model_card.py`.
- `DL/TP8.html` et `DL/TP9.html` corrigés — ils ne comparent plus des notebooks, ils
  documentent et lient les deux `.md`.
- Les deux `model_card.md` sont **inchangés dans leur contenu** (mêmes métriques,
  mêmes sections) — seule la façon de les produire a changé.

**Ce qui ne change pas** : `DL/TP9.ipynb` reste un notebook, à juste titre — son rôle
est une action (entraîner, logger), pas une documentation de référence.